## 01_data_exploration.ipynb

In [2]:
import os
import polars as pl

batch = os.getcwd()
path = os.path.join(batch, '..', 'data', 'kidney_disease.csv')
path = os.path.normpath(path)

# ✅ Tell Polars that ? means missing/null
df = pl.read_csv(
    path,
    null_values=["?", "", "NA", "N/A", "nan", "NaN"],
    infer_schema_length=10000,
    ignore_errors=True
)

print("✅ Loaded Successfully!")
print("Shape        :", df.shape)
print("Null counts  :")
print(df.null_count())
print(df.head())

✅ Loaded Successfully!
Shape        : (400, 26)
Null counts  :
shape: (1, 26)
┌─────┬─────┬─────┬─────┬───┬───────┬─────┬─────┬────────────────┐
│ id  ┆ age ┆ bp  ┆ sg  ┆ … ┆ appet ┆ pe  ┆ ane ┆ classification │
│ --- ┆ --- ┆ --- ┆ --- ┆   ┆ ---   ┆ --- ┆ --- ┆ ---            │
│ u32 ┆ u32 ┆ u32 ┆ u32 ┆   ┆ u32   ┆ u32 ┆ u32 ┆ u32            │
╞═════╪═════╪═════╪═════╪═══╪═══════╪═════╪═════╪════════════════╡
│ 0   ┆ 9   ┆ 12  ┆ 47  ┆ … ┆ 1     ┆ 1   ┆ 1   ┆ 0              │
└─────┴─────┴─────┴─────┴───┴───────┴─────┴─────┴────────────────┘
shape: (5, 26)
┌─────┬──────┬──────┬───────┬───┬───────┬─────┬─────┬────────────────┐
│ id  ┆ age  ┆ bp   ┆ sg    ┆ … ┆ appet ┆ pe  ┆ ane ┆ classification │
│ --- ┆ ---  ┆ ---  ┆ ---   ┆   ┆ ---   ┆ --- ┆ --- ┆ ---            │
│ i64 ┆ f64  ┆ f64  ┆ f64   ┆   ┆ str   ┆ str ┆ str ┆ str            │
╞═════╪══════╪══════╪═══════╪═══╪═══════╪═════╪═════╪════════════════╡
│ 0   ┆ 48.0 ┆ 80.0 ┆ 1.02  ┆ … ┆ good  ┆ no  ┆ no  ┆ ckd            │
│ 1   ┆ 7.0 

## 02_data_cleaning.ipynb

In [3]:
df.null_count()

id,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,bu,sc,sod,pot,hemo,pcv,wc,rc,htn,dm,cad,appet,pe,ane,classification
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,9,12,47,46,49,152,65,4,4,44,19,17,87,88,52,70,105,130,2,2,2,1,1,1,0


In [ ]:
def fill_numeric_with_median(df: pl.DataFrame) -> pl.DataFrame:
    import polars.selectors as cs
    return df.with_columns(
        cs.numeric().fill_null(cs.numeric().median())
    )

df = fill_numeric_with_median(df)

In [7]:
def fill_string_with_mode(df: pl.DataFrame) -> pl.DataFrame:
    import polars.selectors as cs
    return df.with_columns(
        cs.string().fill_null(cs.string().mode())
    )

df = fill_string_with_mode(df)

In [28]:
import polars as pl

num_cols = ["age", "bp", "sg", "al", "su", "bgr", "bu", "sc", 
            "sod", "pot", "hemo", "pcv", "wc", "rc"]

df = df.with_columns([
    pl.col(col)
      .cast(pl.Utf8)                  # ensure string first
      .str.strip_chars()              # remove whitespace/tabs
      .str.replace(r"[^\d.]", "")    # remove non-numeric chars like "?"
      .replace("", None)             # empty string → null
      .cast(pl.Float64)              # now safely cast to numeric ✅
    for col in num_cols
])

# Now median fill works ✅
df = df.with_columns([
    pl.col(col).fill_null(pl.col(col).median())
    for col in num_cols
])

In [29]:
df.null_count()

id,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,bu,sc,sod,pot,hemo,pcv,wc,rc,htn,dm,cad,appet,pe,ane,classification
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


## 03_eda.ipynb

In [30]:
import plotly.express as px

fig = px.box(
    data_frame=df,
    x="age",
    y="bp",
    color="classification",
    boxmode="group",
    notched=True,
    points="all",
    hover_name="classification",
    hover_data={
        "age": True,
        "al": True,
        "bp": True,
    },
    # facet_col="classification",  # ✅ use categorical col, or remove
    orientation="v",
    title="Kidney Disease - Age vs Blood Pressure",
    labels={
        "age": "Age",
        "bp": "Blood Pressure",
        "classification": "CKD Status",
    },
    category_orders={
        "classification": ["ckd", "notckd"]   # ✅ your actual classes
    },
    color_discrete_map={
        "ckd":    "#EF553B",   # ✅ your actual class names
        "notckd": "#636EFA",
    },
    template="plotly_white",
    width=950,
    height=600,
)

fig.update_traces(
    jitter=0.3,
    pointpos=-1.5,
    opacity=0.8,
)

fig.show()

In [31]:
import plotly.express as px

# Box plot — shows outlier dots automatically
fig = px.box(df, y="bp", points="outliers", title="BP Outliers")
fig.show()

# For all numeric columns
import polars.selectors as cs
num_cols = df.select(cs.numeric()).columns

fig = px.box(df.to_pandas()[num_cols], title="All Numeric Outliers")
fig.show()

In [32]:
import polars as pl
import polars.selectors as cs

def remove_outliers_zscore(df: pl.DataFrame, col: str, threshold=3) -> pl.DataFrame:
    mean = df[col].mean()
    std  = df[col].std()
    return df.filter(
        ((pl.col(col) - mean) / std).abs() < threshold
    )

# ✅ Get numeric column names as a list, then loop
num_cols = df.select(cs.numeric()).columns

df_clean = df.clone()
for col in num_cols:
    df_clean = remove_outliers_zscore(df_clean, col=col)

In [33]:
num_cols = df_clean.select(cs.numeric()).columns
fig = px.box(df_clean.to_pandas()[num_cols], title="All Numeric Outliers")
fig.show()

## 04_feature_engineering.ipynb

In [34]:
cat_col = df_clean.select(cs.string()).columns
cat_col

['rbc',
 'pc',
 'pcc',
 'ba',
 'htn',
 'dm',
 'cad',
 'appet',
 'pe',
 'ane',
 'classification']

In [35]:
import polars.selectors as cs

cat_cols = df.select(cs.string()).columns

for col in cat_cols:
    print(f"\n── {col} ──")
    print(df[col].value_counts(sort=True))


── rbc ──
shape: (2, 2)
┌──────────┬───────┐
│ rbc      ┆ count │
│ ---      ┆ ---   │
│ str      ┆ u32   │
╞══════════╪═══════╡
│ normal   ┆ 353   │
│ abnormal ┆ 47    │
└──────────┴───────┘

── pc ──
shape: (2, 2)
┌──────────┬───────┐
│ pc       ┆ count │
│ ---      ┆ ---   │
│ str      ┆ u32   │
╞══════════╪═══════╡
│ normal   ┆ 324   │
│ abnormal ┆ 76    │
└──────────┴───────┘

── pcc ──
shape: (2, 2)
┌────────────┬───────┐
│ pcc        ┆ count │
│ ---        ┆ ---   │
│ str        ┆ u32   │
╞════════════╪═══════╡
│ notpresent ┆ 358   │
│ present    ┆ 42    │
└────────────┴───────┘

── ba ──
shape: (2, 2)
┌────────────┬───────┐
│ ba         ┆ count │
│ ---        ┆ ---   │
│ str        ┆ u32   │
╞════════════╪═══════╡
│ notpresent ┆ 378   │
│ present    ┆ 22    │
└────────────┴───────┘

── htn ──
shape: (2, 2)
┌─────┬───────┐
│ htn ┆ count │
│ --- ┆ ---   │
│ str ┆ u32   │
╞═════╪═══════╡
│ no  ┆ 253   │
│ yes ┆ 147   │
└─────┴───────┘

── dm ──
shape: (5, 2)
┌──────┬───────┐
│ dm

In [24]:
df.null_count()

id,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,bu,sc,sod,pot,hemo,pcv,wc,rc,htn,dm,cad,appet,pe,ane,classification
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,70,105,130,0,0,0,0,0,0,0


In [27]:
df.null_count()


id,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,bu,sc,sod,pot,hemo,pcv,wc,rc,htn,dm,cad,appet,pe,ane,classification
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
